In [2]:
!mamba install pandas numpy scikit-learn matplotlib

mambajs 0.19.13

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas, scikit-learn
Channels: emscripten-forge, conda-forge

Solving environment...
Solving took 22.299200000047684 seconds
  Name                          Version                       Build                         Channel                       
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
+ brotli-python                 1.2.0                         py313h33caa6c_0               emscripten-forge              
+ certifi                       2026.4.22                     pyhd8ed1ab_0                  conda-forge                   
+ charset-normalizer            3.4.7                         pyhd8ed1ab_0                  conda-forge                   
+ idna                          3.13                          pyhcf101f3_0                  conda-forge                   
+ joblib                        1.5.3

In [3]:
# ==========================================
# PASSO 1: Importação de Bibliotecas e Dados
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

print("Carregando dados...")
np.random.seed(42)
df = pd.DataFrame(np.random.randn(1000, 29), columns=[f'V{i}' for i in range(1, 29)] + ['Amount'])
df['Class'] = np.random.choice([0, 1], size=1000, p=[0.99, 0.01]) 
df['Time'] = np.arange(1000)

print("Primeiras linhas do dataset:")
print(df.head())

# ==========================================
# PASSO 2: Análise de Desbalanceamento
# ==========================================
print("\nDistribuição das classes (Desbalanceamento):")
print(df['Class'].value_counts(normalize=True))

# ==========================================
# PASSO 3: Feature Engineering
# ==========================================
print("\nAplicando Feature Engineering...")
df['Amount_log'] = np.log1p(df['Amount'].clip(lower=0)) 

scaler = StandardScaler()
df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])

# ==========================================
# PASSO 4: Preparação e Separação dos Dados
# ==========================================
X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print(f"\nTamanho do treino: {X_train.shape[0]} | Tamanho do teste: {X_test.shape[0]}")

# ==========================================
# PASSO 5: Regressão Logística (Baseline)
# ==========================================
print("\n--- Treinando Regressão Logística (Baseline) ---")
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

print("Relatório - Regressão Logística:")
print(classification_report(y_test, y_pred_lr, zero_division=0))

# ==========================================
# PASSO 6: Random Forest e Ajuste de Threshold
# ==========================================
print("\n--- Treinando Random Forest ---")
rf_model = RandomForestClassifier(n_estimators=50, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print("Relatório - Random Forest:")
print(classification_report(y_test, y_pred_rf, zero_division=0))

print("\n--- Ajuste de Threshold (Limiar) na Regressão Logística ---")
y_pred_proba_lr = lr_model.predict_proba(X_test)[:, 1]
threshold_custom = 0.3
y_pred_custom = (y_pred_proba_lr > threshold_custom).astype(int)

print(f"Relatório com Threshold = {threshold_custom}:")
print(classification_report(y_test, y_pred_custom, zero_division=0))

print("\n🚀 Modelos processados com sucesso no navegador!")

Carregando dados...
Primeiras linhas do dataset:
         V1        V2        V3        V4        V5        V6        V7  \
0  0.496714 -0.138264  0.647689  1.523030 -0.234153 -0.234137  1.579213   
1 -0.291694 -0.601707  1.852278 -0.013497 -1.057711  0.822545 -1.220844   
2  0.331263  0.975545 -0.479174 -0.185659 -1.106335 -1.196207  0.812526   
3  0.328751 -0.529760  0.513267  0.097078  0.968645 -0.702053 -0.327662   
4 -0.034712 -1.168678  1.142823  0.751933  0.791032 -0.909387  1.402794   

         V8        V9       V10  ...       V22       V23       V24       V25  \
0  0.767435 -0.469474  0.542560  ... -0.225776  0.067528 -1.424748 -0.544383   
1  0.208864 -1.959670 -1.328186  ...  0.324084 -0.385082 -0.676922  0.611676   
2  1.356240 -0.072010  1.003533  ... -1.987569 -0.219672  0.357113  1.477894   
3 -0.392108 -1.463515  0.296120  ...  0.257550 -0.074446 -1.918771 -0.026514   
4 -1.401851  0.586857  2.190456  ... -0.322062  0.813517 -1.230864  0.227460   

        V26       V